# 开发环境与样例运行
上一节带你看清了 PyAsc 工程目录与课程随附的算子样例。本节正式动手，把 PyAsc 开发环境从 0 搭起来，并端到端跑通第一个样例，验证环境是否正常。本节学习大纲如下。
- 软硬件配套关系与开发环境验证
- PyAsc 两种安装方式：pip 快速安装 vs 源码编译安装
- 运行环境变量配置与运行参数
- 端到端跑通 `01_add` 样例

---


## 1. 软硬件配套关系与开发环境验证

### 1.1 软硬件配套关系
PyAsc 不同发行版可支持的 CANN 版本及昇腾产品如下：

| pyasc 社区版本 | 支持 CANN 包版本 | 支持昇腾产品 |
| --- | --- | --- |
| v1.1.0、v1.1.1 | 社区版 8.5.0.alpha001 及以上 | Atlas A2 训练/推理产品（910B）、Atlas A3 训练/推理产品（910C） |
| v1.0.0 | 社区版 8.5.0.alpha001、8.5.0.alpha002 | Atlas A2 训练/推理产品（910B）、Atlas A3 训练/推理产品（910C） |

软件依赖：

- python：3.9 ≤ version ≤ 3.12
- gcc / g++：≥ 9.4.0（且 gcc 与 g++ 版本一致）
- GLIBC：≥ 2.31
- cmake：≥ 3.20

### 1.2 开发环境验证
完成 CANN 安装后，先验证驱动与 CANN 是否就位。


In [ ]:
# 检查 NPU 驱动
!npu-smi info


---
## 2. PyAsc 安装：pip 快速安装 vs 源码编译安装
PyAsc 支持两种安装方式，**选择哪一种取决于你后续要做什么**：

| 方式 | 命令 | 使用场景 |
| --- | --- | --- |
| pip 快速安装 | `pip install pyasc` | 仅使用 PyAsc 已有 API 写算子、跑算子，不修改 PyAsc 源码 |
| 源码编译安装 | 下载 LLVM + `pip install .` 或 `pip install -e .` | 需要新增/修改 PyAsc 接口、扩展 ASC-IR 或 Ascend C 代码生成、向 PyAsc 上游贡献代码 |

#

> ✨ 本章及"使用 PyAsc 写算子"类章节的样例运行只需 **pip 快速安装**；"PyAsc 接口扩展"类章节必须使用 **源码编译安装**（推荐 `pip install -e .` 开发者模式）。

#

### 2.1 pip 快速安装
二进制 wheel 安装包支持 CPython 3.9–3.12。本课程后续样例运行流程默认按此方式介绍。


In [ ]:
# 安装 PyAsc 稳定版（CPython 3.9–3.12）
!python3 -m pip install pyasc
!python3 -m pip list 2>/dev/null | grep -w pyasc || echo '[WARN] pyasc 未安装，请检查上面的安装日志'


### 2.2 源码编译安装
源码编译需要先准备 LLVM。

**Step 1：下载 LLVM 预编译包**

PyAsc 后端基于 MLIR/LLVM。请根据 CPU 架构（aarch64 / x86_64）选择对应包。

```bash
# ARM
wget <vendor 提供的 llvm-19.1.7-aarch64.tar.xz 下载链接>
tar -xJf llvm-19.1.7-aarch64.tar.xz
export LLVM_INSTALL_PREFIX=$PWD/llvm-19.1.7-aarch64

# x86
wget <vendor 提供的 llvm-19.1.7-x86_64.tar.xz 下载链接>
tar -xJf llvm-19.1.7-x86_64.tar.xz
export LLVM_INSTALL_PREFIX=$PWD/llvm-19.1.7-x86_64
```

LLVM 依赖系统库 `libz`、`libzstd`，安装前请确保已通过包管理器安装（缺失时可执行 `sudo apt-get install zlib1g-dev libzstd-dev`）。

**Step 2：构建 PyAsc**

进入 PyAsc 源码根目录，按如下命令构建：

```bash
cd <pyasc 源码根目录>
export LLVM_INSTALL_PREFIX=<llvm_install_path>

# 普通安装（拷贝到 site-packages，本地修改不生效）
python3 -m pip install .

# 开发者模式（符号链接，本地修改实时生效，推荐用于接口扩展开发）
python3 -m pip install -e .
```

### 2.3 安装验证


In [ ]:
# 若能输出 pyasc 及版本号，则说明安装成功
!python3 -m pip list 2>/dev/null | grep -w pyasc || echo '[WARN] pyasc 尚未安装，请按上文步骤完成 pip 或源码安装'


---
## 3. 运行环境变量配置与运行参数

### 3.1 source CANN 环境变量
运行 PyAsc 算子前，需要 `source` CANN 环境变量。在线体验环境与 CANN 官方 Docker 镜像已自动配置，可跳过；手动安装用户必须执行。

```bash
# 默认路径（root 用户）
source /usr/local/Ascend/cann/set_env.sh
# 非 root 用户：将 /usr/local 替换为 ${HOME}
# 自定义路径：source ${install_path}/cann/set_env.sh
```

如果环境中安装了多个版本的 CANN，请确保 `set_env.sh` 路径指向的是与 PyAsc 配套的版本目录。

### 3.2 运行参数：`-r RUN_MODE` 与 `-v SOC_VERSION`
所有 tutorials 样例都遵循同一套命令行接口：

```bash
python3 <sample>.py -r [RUN_MODE] -v [SOC_VERSION]
```

参数含义：

| 参数 | 含义 | 取值 |
| --- | --- | --- |
| `-r RUN_MODE` | 编译执行方式 | `NPU`（上板） / `Model`（仿真器） |
| `-v SOC_VERSION` | 昇腾 AI 处理器型号 | 如 `Ascend910B1`、`Ascend910B4`、`Ascend910C1` 等 |

本课程默认使用 **NPU 上板模式**（`-r NPU`）。NPU 上板模式不需要额外配置仿真器相关库，仅需 `source set_env.sh` 即可执行。

如需查询 `SOC_VERSION`，在 NPU 主机上执行 `npu-smi info`，在输出表头的 `Name` 字段前加 `Ascend` 前缀即可，例如 `Name=910B4` 对应 `-v Ascend910B4`。

> 仿真器模式（`-r Model`）需要额外设置 `LD_LIBRARY_PATH=$ASCEND_HOME_PATH/tools/simulator/Ascend910B1/lib:$LD_LIBRARY_PATH`，并在接入 torch_npu 时通过 `export LD_PRELOAD=libruntime_camodel.so` 预加载仿真动态库。本课程不展开仿真器模式的细节。


---
## 4. 端到端跑通 `01_add`
环境配置完成后，使用 NPU 上板模式跑一遍 `01_add`，确认 PyAsc → ASC-IR → Ascend C → NPU 二进制 这条链路是通的。下面以 `Ascend910B1` 为例（请根据实际 SOC 替换 `-v` 取值）。


In [ ]:
!python3 ./src/tutorials/01_add/add.py -r NPU -v Ascend910B1


如果一切正常，最后一行会出现：

```
INFO:root:[INFO] Sample add run success.
```

这就说明：

1. PyAsc 安装正确。
2. CANN 环境变量配置正确。
3. NPU 后端可用。
4. PyAsc → ASC-IR → Ascend C → NPU 执行 这条链路是通的。


---

环境配置与样例运行都已完成。下一节会带你进入本章综合实践，独立完成环境搭建并跑通课程提供的样例。请继续学习 [章节实践](01.05_chapter_practice.ipynb)。


---
## 课后练习
本节介绍了 PyAsc 的安装方式、运行环境变量配置、运行参数与样例运行命令，请通过以下题目检验掌握情况。

1. （单选题）下列场景中应使用源码编译安装 PyAsc 的是？  
    A. 仅使用 PyAsc 已有 API 写算子、跑算子  
    B. 需要新增/修改 PyAsc 接口、扩展 ASC-IR 或 Ascend C 代码生成  
    C. 仅在 NPU 上板模式下运行已有样例  
    D. 只想升级 PyAsc 到最新稳定版  

2. （多选题）PyAsc 源码编译安装所需的基础依赖包含哪些？  
    A. Python 3.9–3.12  
    B. gcc / g++ ≥ 9.4.0（版本一致）  
    C. cmake ≥ 3.20  
    D. PyTorch ≥ 2.7  

3. （单选题）所有课程样例都通过哪两个命令行参数控制运行方式？  
    A. `--mode` 与 `--device`  
    B. `-r` 与 `-v`  
    C. `--backend` 与 `--platform`  
    D. `-m` 与 `-s`  

4. （单选题）在 NPU 上板模式下跑通 PyAsc 样例，需要执行的环境变量配置是？  
    A. `source /usr/local/Ascend/cann/set_env.sh`  
    B. `export LD_PRELOAD=libruntime_camodel.so`  
    C. `export LD_LIBRARY_PATH=$ASCEND_HOME_PATH/tools/simulator/Ascend910B1/lib:$LD_LIBRARY_PATH`  
    D. 无需任何环境变量配置  

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/01.04_answer.txt
